# 03. DCA-Trie v2: Iterative Symbolic Expansion

**Purpose:** Implement and evaluate DCA-Trie v2 — step-wise dynamic trie expansion using symbolic gates.

**Method:** Start with a minimal trie (question entities only). At each entity commit, fetch all neighbours from the KG and admit only those edges that pass both symbolic gates. The trie grows incrementally.

**Reference:** Chapter 3, §3.7, Algorithm 2.

**Key difference from v1:** v2 never enumerates all paths upfront. It expands one hop at a time as entities are committed.

In [ ]:
# Setup & Environment

import sys
import os
import subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print("Not on Colab — skipping Drive mount.")

import torch

# GPU detection
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    sm_arch = torch.cuda.get_device_capability(0)
    sm_str = f"SM {sm_arch[0]}.{sm_arch[1]}"
    has_t4 = "T4" in gpu_name
    has_l4 = "L4" in gpu_name
    has_a100 = "A100" in gpu_name
    has_h100 = "H100" in gpu_name
else:
    gpu_name = "None"
    gpu_mem = 0
    sm_str = "N/A"
    has_t4 = has_l4 = has_a100 = has_h100 = False

print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  {sm_str}")

# Pinned dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "datasets==3.1.0",
    "marisa-trie==1.2.0",
    "networkx==3.3",
    "scikit-learn==1.5.2",
    "tiktoken==0.8.0",
    "sentencepiece==0.2.0",
    "protobuf==5.28.3",
    "python-dotenv==1.0.1",
], check=True)

# Flash Attention 2 install (3-tier fallback)
flash_attn_installed = False
try:
    import flash_attn
    flash_attn_installed = True
    print(f"flash-attn already installed (v{flash_attn.__version__}).")
except ImportError:
    print("flash-attn not found. Attempting install...")
    wheel_url = (
        f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/"
        f"flash_attn-2.6.3+cu{''.join(map(str, torch.version.cuda.split('.')[:2]))}"
        f"-cp{sys.version_info.major}{sys.version_info.minor}-"
        f"cp{sys.version_info.major}{sys.version_info.minor}-"
        f"linux_x86_64.whl"
    )
    ret = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", wheel_url],
        capture_output=True,
    )
    if ret.returncode == 0:
        flash_attn_installed = True
        print("flash-attn installed from wheel.")
    else:
        print("Wheel install failed. Trying source build...")
        ret2 = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "flash-attn==2.6.3", "--no-build-isolation"],
            capture_output=True,
        )
        if ret2.returncode == 0:
            flash_attn_installed = True
            print("flash-attn installed from source.")
        else:
            print("flash-attn install failed. Falling back to sdpa.")

# Clone repo if needed
if not os.path.exists("graph-constrained-reasoning"):
    subprocess.run([
        "git", "clone",
        "https://github.com/Adjanour/graph-constrained-reasoning-dca",
    ], check=True)

os.chdir("graph-constrained-reasoning")
sys.path.insert(0, ".")

# Imports
import json
import copy
import re
import warnings
import argparse
from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from collections import defaultdict
import networkx as nx

from src.llms import get_registed_model
from src.trie import MarisaTrie
from src.graph_constrained_decoding import GraphConstrainedDecoding
import src.utils as utils

sys.path.insert(0, os.path.join(os.getcwd(), "notebooks"))
from type_oracle import TypeOracle

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
print("Imports ready.")

## Configuration

In [ ]:
# Model
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# Dataset
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# Decoding (v2 uses step-wise greedy since trie grows one hop at a time)
INDEX_LEN = 2
K = 5
GEN_MODE = "greedy"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256

# Attention implementation
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

# Sampling
MAX_SAMPLES = 100  # set to None for full dataset

# Output
PREDICT_PATH = "results/GenPaths"
FORCE_RERUN = True

# Google Drive (Colab only)
DRIVE_BASE = "/content/drive/MyDrive/GCR-experiments" if ON_COLAB else "results"

tag = "DCAv2-symbolic"
print(f"Model: {MODEL_PATH}")
print(f"Dataset: {DATASET}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")
print(f"Tag: {tag}")
print(f"ATTN_IMPL: {ATTN_IMPL}")
print(f"Drive base: {DRIVE_BASE}")

## Load LLM

In [ ]:
parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

try:
    print("Loading LLM...")
    model = LLM(args)
    model.prepare_for_inference()
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
    print("Ready.")
except Exception as e:
    print(f"Failed to load model: {e}")
    raise

## DCA-Trie v2: Iterative Symbolic Expansion

In [ ]:
def build_path_trie(tokenizer, path_strings):
    """Build a MarisaTrie from a list of path strings."""
    if len(path_strings) == 0:
        return None
    tokenized = tokenizer(
        path_strings, padding=False, add_special_tokens=False
    ).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    return MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)


def extract_entity_from_output(output_text, end_token="</PATH>"):
    """Extract the last committed entity from the generated text."""
    cleaned = output_text.replace(end_token, "").strip()
    parts = cleaned.split(" -> ")
    if len(parts) >= 1:
        return parts[-1].strip()
    return None


def dca_v2_generate_symbolic(
    question, start_entities, graph, nx_graph, llm_model,
    tokenizer, oracle, max_hops=2,
):
    """
    DCA-Trie v2 generation loop (Algorithm 2, section 3.7.2).

    At each entity commit:
      1. Check if terminal hop reached
      2. Fetch neighbours of committed entity from graph
      3. Gate each neighbour through range_gate (all hops) + type_gate (terminal)
      4. Expand trie with admitted edges (Eq. 3.16)

    No embeddings, no threshold, no residual vector.
    """
    answer_types = oracle.infer_answer_types(question)
    start_token = "<PATH>"
    end_token = "</PATH>"
    start_id = tokenizer.convert_tokens_to_ids(start_token)
    end_id = tokenizer.convert_tokens_to_ids(end_token)

    # Step 1: Initialise trie with first-hop gated neighbours
    first_hop_paths = []
    for entity in start_entities:
        if entity not in nx_graph:
            continue
        for neighbor in nx_graph.neighbors(entity):
            rel = nx_graph[entity][neighbor]["relation"]
            if not oracle.range_gate(rel, neighbor):
                continue
            path_str = f"{entity} -> {rel} -> {neighbor}"
            first_hop_paths.append(path_str)

    if len(first_hop_paths) == 0:
        return None

    current_trie = build_path_trie(tokenizer, first_hop_paths)
    if current_trie is None:
        return None

    # Build initial prompt
    prompt = (
        "Reasoning path is a sequence of triples in the KG that connects the topic entities "
        "to answer entities. Given the question, generate reasoning paths starting from "
        "the topic entities to answer the question.\n\n"
        f"# Question:\n{question}\n"
        f"# Topic entities:\n{', '.join(start_entities)}\n"
    )

    output_text = ""
    committed_entity = start_entities[0] if start_entities else None
    hop = 0

    for step in range(max_hops * 3):
        # Constrained generation step
        llm_input = llm_model.prepare_model_prompt(prompt)
        gcr = GraphConstrainedDecoding(
            tokenizer, current_trie, start_id, end_id,
            enable_constrained_by_default=False,
        )
        inputs = tokenizer(
            llm_input, return_tensors="pt", add_special_tokens=False
        )
        input_ids = inputs.input_ids.to(llm_model.model.device)
        attn_mask = inputs.attention_mask.to(llm_model.model.device)

        res = llm_model.model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            generation_config=llm_model.generation_cfg,
            prefix_allowed_tokens_fn=gcr.allowed_tokens_fn,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=32,
        )

        output = tokenizer.decode(
            res.sequences[0][input_ids.shape[1]:],
            skip_special_tokens=True,
        )
        output_text += output

        if end_token in output or tokenizer.eos_token in output:
            break

        # Extract committed entity
        new_entity = extract_entity_from_output(output, end_token)
        if new_entity is None or new_entity == committed_entity:
            continue
        committed_entity = new_entity
        hop += 1

        if hop >= max_hops:
            break

        # Expand trie with gated neighbours (Algorithm 2, lines 17-30)
        is_terminal = (hop + 1 >= max_hops)
        new_paths = []

        if committed_entity in nx_graph:
            for neighbor in nx_graph.neighbors(committed_entity):
                rel = nx_graph[committed_entity][neighbor]["relation"]

                # Range gate (Eq. 3.13) — must pass at every hop
                if not oracle.range_gate(rel, neighbor):
                    continue

                # Type gate (Eq. 3.12) — terminal hop only
                if is_terminal and not oracle.type_gate(
                    neighbor, answer_types, hop + 1, max_hops
                ):
                    continue

                path_str = f"{committed_entity} -> {rel} -> {neighbor}"
                new_paths.append(path_str)

        if len(new_paths) > 0:
            expanded_trie = build_path_trie(tokenizer, new_paths)
            if expanded_trie is not None:
                current_trie = expanded_trie
                prompt = prompt + f"\n{output.strip()}\n"
        else:
            break

    return output_text


# Run inference
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")
else:
    print(f"Loaded {len(dataset)} questions")

model_short = MODEL_PATH.split("/")[-1]
postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
output_dir = os.path.join(PREDICT_PATH, DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

predictions_path = os.path.join(output_dir, "predictions.jsonl")

if FORCE_RERUN or not os.path.exists(predictions_path):
    processed_ids = set()
    file_mode = "w"
else:
    with open(predictions_path) as f:
        processed_ids = {json.loads(line)["id"] for line in f}
    file_mode = "a"
    print(f"Resuming: {len(processed_ids)} predictions already on disk.")

n_expansion_dead_ends = 0
n_errors = 0

with open(predictions_path, file_mode) as fout:
    for data in tqdm(dataset, desc="DCA-Trie v2 (Symbolic)"):
        qid = data["id"]
        if qid in processed_ids:
            continue

        graph = data["graph"]
        nx_graph = utils.build_graph(graph)
        oracle = TypeOracle.from_graph(graph)
        truth_paths = utils.get_truth_paths(
            data["q_entity"], data["a_entity"], nx_graph
        )
        ground_paths = [utils.path_to_string(p) for p in truth_paths]

        try:
            prediction = dca_v2_generate_symbolic(
                question=data["question"],
                start_entities=data["q_entity"],
                graph=graph,
                nx_graph=nx_graph,
                llm_model=model,
                tokenizer=model.tokenizer,
                oracle=oracle,
                max_hops=INDEX_LEN,
            )
        except Exception as e:
            n_errors += 1
            print(f"Error on {qid}: {e}")
            continue

        if prediction is None:
            n_expansion_dead_ends += 1
            continue

        fout.write(json.dumps({
            "id": qid,
            "question": data["question"],
            "prediction": prediction,
            "ground_truth": data["answer"],
            "ground_truth_paths": ground_paths,
            "dca_version": "v2",
            "filtering": "symbolic",
        }) + "\n")
        fout.flush()

print(f"\nExpansion dead ends: {n_expansion_dead_ends}")
print(f"Errors: {n_errors}")
print("Done.")

## Compare with GCR and DCA-Trie v1

In [ ]:
print(
    "Comparison: GCR vs DCA-Trie v1 vs DCA-Trie v2\n"
    "\n"
    "Metric               GCR         DCA-v1       DCA-v2 (symbolic)\n"
    "------               ---         ------       -----------------\n"
    "Hits@1               89.0 (dev)  ?            ?\n"
    "F1                   ?           ?            ?\n"
    "Faithfulness        100%        100%         100%\n"
    "SIR*                 high        ?            ?\n"
    "SIR*_type            high        ~0           ~0\n"
    "SIR*_traj            high        ?            ?\n"
    "\n"
    "Expected: both DCA variants << GCR on SIR metrics.\n"
    "v2 should have marginally lower SIR*_traj than v1 because\n"
    "it only gates actual neighbours of the committed entity,\n"
    "not all BFS paths pre-computed."
)